In [1]:
import re
import json
import copy
import random
import time
from pathlib import Path
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import warnings
warnings.filterwarnings("ignore")

PROJECT_ROOT  = Path.home() / "icidea_llm_ids"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
RESULTS_DIR   = PROJECT_ROOT / "results"
ROAD_DIR      = PROJECT_ROOT / "data" / "ROAD"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

def parse_road_log(filepath, max_rows=500000):
    """
    Parse ROAD candump log format:
    (timestamp) can0 CAN_ID#DATA_HEX
    """
    rows = []
    pattern = re.compile(
        r'\((\d+\.\d+)\)\s+\w+\s+([0-9A-Fa-f]+)#([0-9A-Fa-f]*)'
    )
    with open(filepath, "r") as f:
        for i, line in enumerate(f):
            if i >= max_rows:
                break
            m = pattern.match(line.strip())
            if not m:
                continue
            ts_str, can_id_str, data_str = m.groups()
            try:
                ts     = float(ts_str)
                can_id = int(can_id_str, 16)
                # Parse data bytes
                data_hex = data_str
                n_bytes  = len(data_hex) // 2
                dlc      = min(n_bytes, 8)
                data = []
                for j in range(dlc):
                    data.append(int(data_hex[j*2:j*2+2], 16))
                # Pad to 8 bytes
                data += [0] * (8 - dlc)
                rows.append({
                    "timestamp": ts,
                    "can_id":    can_id,
                    "dlc":       dlc,
                    **{f"data{k}": data[k] for k in range(8)},
                })
            except (ValueError, IndexError):
                continue
    return pd.DataFrame(rows)

# Test parser
print("Testing parser on ambient log...")
test_df = parse_road_log(ROAD_DIR / "ambient_dyno_drive_basic_short.log",
                          max_rows=1000)
print(f"✓ Parsed {len(test_df)} rows")
print(f"  Columns: {list(test_df.columns)}")
print(f"  CAN ID range: {test_df['can_id'].min()} - {test_df['can_id'].max()}")
print(f"  DLC range: {test_df['dlc'].min()} - {test_df['dlc'].max()}")
print(f"\nSample rows:")
print(test_df.head(3).to_string())

/Users/deepakpatnaik/icidea_llm_ids/venv/lib/python3.11/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


Testing parser on ambient log...
✓ Parsed 1000 rows
  Columns: ['timestamp', 'can_id', 'dlc', 'data0', 'data1', 'data2', 'data3', 'data4', 'data5', 'data6', 'data7']
  CAN ID range: 14 - 4095
  DLC range: 8 - 8

Sample rows:
      timestamp  can_id  dlc  data0  data1  data2  data3  data4  data5  data6  data7
0  1.030000e+09     354    8      0      8      0      3    234     17    244    206
1  1.030000e+09     167    8      0     16    250     39      1     43    144    160
2  1.030000e+09    4095    8      0      0      0      0      0      0      0      0


In [2]:
print("Loading ROAD dataset files...")

# Load each file with label
road_files = {
    "AMBIENT":      (ROAD_DIR / "ambient_dyno_drive_basic_short.log",    0),
    "ACCELERATOR":  (ROAD_DIR / "accelerator_attack_drive_1.log",        1),
    "FUZZING":      (ROAD_DIR / "fuzzing_attack_1.log",                  2),
    "SPEEDOMETER":  (ROAD_DIR / "max_speedometer_attack_1.log",          3),
    "CORRELATED":   (ROAD_DIR / "correlated_signal_attack_1.log",        4),
}

ROAD_LABEL_MAP = {
    0: "AMBIENT",
    1: "ACCELERATOR",
    2: "FUZZING",
    3: "SPEEDOMETER",
    4: "CORRELATED"
}

road_dfs = {}
for name, (path, label) in road_files.items():
    df = parse_road_log(path, max_rows=300000)
    df["label"] = label
    df["label_name"] = name
    road_dfs[name] = df
    print(f"  {name:15s}  rows={len(df):,}  "
          f"unique_ids={df['can_id'].nunique()}")

print(f"\n✓ All files loaded")

Loading ROAD dataset files...
  AMBIENT          rows=300,000  unique_ids=106
  ACCELERATOR      rows=204,760  unique_ids=106
  FUZZING          rows=49,342  unique_ids=662
  SPEEDOMETER      rows=213,397  unique_ids=106
  CORRELATED       rows=81,262  unique_ids=106

✓ All files loaded


In [3]:
WINDOW_SIZE = 14
STRIDE      = 5
SAMPLES_PER_CLASS = 50  # 50 windows per class = 250 total

def frames_to_text(frames):
    n = len(frames)
    lines = [f"CAN Bus Telemetry Sequence ({n} frames):"]
    for i, frame in enumerate(frames, start=1):
        can_id_hex = f"{frame['can_id']:04X}"
        dlc = frame["dlc"]
        data_bytes = frame["data"][:dlc]
        data_hex = " ".join(f"{b:02X}" for b in data_bytes)
        ts = frame["timestamp"]
        lines.append(
            f"[{i:03d}] T={ts:.3f} ID={can_id_hex} "
            f"DLC={dlc} DATA={data_hex}"
        )
    return "\n".join(lines)

def generate_windows_from_df(df, window_size, stride):
    """Generate windows from a sorted DataFrame."""
    df = df.sort_values("timestamp").reset_index(drop=True)
    windows = []
    n = len(df)
    for start in range(0, n - window_size + 1, stride):
        chunk = df.iloc[start:start + window_size]
        frames = []
        for _, row in chunk.iterrows():
            frames.append({
                "timestamp": float(row["timestamp"]),
                "can_id":    int(row["can_id"]),
                "dlc":       int(row["dlc"]),
                "data":      [int(row[f"data{i}"]) for i in range(8)],
            })
        windows.append({
            "frames":    frames,
            "text":      frames_to_text(frames),
            "label":     int(df.iloc[start]["label"]),
            "label_name": str(df.iloc[start]["label_name"]),
        })
    return windows

# Generate and sample windows per class
print("Generating windows...")
all_road_windows = []

for name, df in road_dfs.items():
    windows = generate_windows_from_df(df, WINDOW_SIZE, STRIDE)
    # Sample
    np.random.seed(SEED)
    n = min(SAMPLES_PER_CLASS, len(windows))
    idxs = np.random.choice(len(windows), n, replace=False)
    sampled = [windows[i] for i in idxs]
    all_road_windows.extend(sampled)
    print(f"  {name:15s}  total_windows={len(windows):,}  "
          f"sampled={n}")

print(f"\n✓ Total ROAD windows: {len(all_road_windows)}")

# Convert to DataFrame
road_windows_df = pd.DataFrame([{
    "label":      w["label"],
    "label_name": w["label_name"],
    "text":       w["text"],
    "frames_json": json.dumps(w["frames"]),
} for w in all_road_windows])

print(f"\nClass distribution:")
print(road_windows_df["label_name"].value_counts().to_string())

# Save for Kaggle
save_path = ARTIFACTS_DIR / "task4_road_windows.parquet"
road_windows_df.to_parquet(save_path, index=False)
size_mb = save_path.stat().st_size / 1e6
print(f"\n✓ Saved to {save_path} ({size_mb:.2f} MB)")

Generating windows...
  AMBIENT          total_windows=59,998  sampled=50
  ACCELERATOR      total_windows=40,950  sampled=50
  FUZZING          total_windows=9,866  sampled=50
  SPEEDOMETER      total_windows=42,677  sampled=50
  CORRELATED       total_windows=16,250  sampled=50

✓ Total ROAD windows: 250

Class distribution:
label_name
AMBIENT        50
ACCELERATOR    50
FUZZING        50
SPEEDOMETER    50
CORRELATED     50

✓ Saved to /Users/deepakpatnaik/icidea_llm_ids/artifacts/task4_road_windows.parquet (0.16 MB)
